# Meme Decoder Colab Pipeline

This notebook runs the MemeCap ablation pipeline on Google Colab:

- Input settings `1-5`
- Training strategies: `zero-shot`, `projector-only`, `projector-lora`
- Google Drive data/cache/output paths

Recommended first run: use the smoke-test cells with small sample counts before launching full training.

## 1. Runtime

In Colab, choose **Runtime -> Change runtime type -> GPU**. L4/A100 is best; T4 can be used with batch size 1 and gradient accumulation.

In [3]:
## 1. Runtime

In Colab, choose **Runtime -> Change runtime type -> GPU**. A100 is ideal for this project.

The commands below assume your meme images are already uploaded to Google Drive, so they do not download images from URLs by default.


Mounted at /content/drive


In [4]:
# Basic GPU check
!nvidia-smi

Thu Apr 30 06:13:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Or Update The Repository

In [9]:
import os
from pathlib import Path

REPO_URL = "https://github.com/baoyunfan0101/meme-decoder.git"
REPO_DIR = Path("/content/meme-decoder")

if REPO_DIR.exists():
    %cd /content/meme-decoder
    !git pull
else:
    %cd /content
    !git clone $REPO_URL
    %cd /content/meme-decoder

/content/meme-decoder
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.10 KiB | 2.10 MiB/s, done.
From https://github.com/baoyunfan0101/meme-decoder
   78bef9d..7869688  main       -> origin/main
Updating 78bef9d..7869688
Fast-forward
 pipeline_colab.ipynb | 112 ++++++++++++++++++++++++++++++++++++++++++++++-----
 requirements.txt     |   2 +-
 2 files changed, 104 insertions(+), 10 deletions(-)


## 3. Install Dependencies

In [11]:
%cd /content/meme-decoder
!pip install -q -r requirements.txt

# Keep the VLM stack current. bitsandbytes is useful if you later enable 4-bit loading.
!pip install -q -U transformers accelerate peft bitsandbytes


/content/meme-decoder
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 138.3 MB/s eta 0:00:0000:0100:01


## 4. Configure Data Paths

Expected data layout in Drive:

```text
/content/drive/MyDrive/meme-decoder-data/data/
  raw/
    memes-trainval.json
    memes-test.json
    memes/
      memes/
        memes/
          memes_bpet7l.png
          ...
  processed/
    memes-trainval.ocr.json
    memes-test.ocr.json
```

Set `RAW_DIR` to `data/raw`, not to the deepest image folder. `src/path_utils.py` automatically descends into nested `memes/` folders when locating images.

Because images are already uploaded to Drive, the training/evaluation commands below do **not** use `--allow-download`.


In [12]:
from pathlib import Path
import os

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/meme-decoder-data")
DATA_DIR = DRIVE_PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
IMAGE_CACHE_DIR = RAW_DIR / "memes" / "memes" / "memes"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["RAW_DIR"] = str(RAW_DIR)
os.environ["PROCESSED_DIR"] = str(PROCESSED_DIR)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)

print("RAW_DIR        =", RAW_DIR)
print("IMAGE_CACHE_DIR=", IMAGE_CACHE_DIR)
print("PROCESSED_DIR  =", PROCESSED_DIR)
print("OUTPUT_DIR     =", OUTPUT_DIR)
print("raw exists      :", RAW_DIR.exists())
print("image dir exists:", IMAGE_CACHE_DIR.exists())
print("processed exists:", PROCESSED_DIR.exists())


RAW_DIR      = /content/drive/MyDrive/meme-decoder-data/data/raw
PROCESSED_DIR= /content/drive/MyDrive/meme-decoder-data/data/processed
OUTPUT_DIR   = /content/drive/MyDrive/meme-decoder-data/outputs


## 5. Verify Drive Data

This notebook expects the data to already be in Google Drive. It will not download the dataset by default.


In [ ]:
required_paths = [
    RAW_DIR / "memes-trainval.json",
    RAW_DIR / "memes-test.json",
    PROCESSED_DIR / "memes-trainval.ocr.json",
    PROCESSED_DIR / "memes-test.ocr.json",
    IMAGE_CACHE_DIR,
]

for path in required_paths:
    print(f"{str(path):95s} exists={path.exists()}")

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Some required Drive paths are missing. Check the paths printed above.")


## 6. Verify Dataset Counts

In [13]:
import json

for path in [PROCESSED_DIR / "memes-trainval.ocr.json", PROCESSED_DIR / "memes-test.ocr.json"]:
    print(path)
    if not path.exists():
        print("  MISSING")
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    print("  count:", len(data))
    print("  keys :", sorted(data[0].keys()) if data else [])

/content/drive/MyDrive/meme-decoder-data/data/processed/memes-trainval.ocr.json
  count: 5823
  keys : ['category', 'img_captions', 'img_fname', 'meme_captions', 'metaphors', 'ocr_confidence', 'ocr_engine', 'ocr_text', 'post_id', 'title', 'url']
/content/drive/MyDrive/meme-decoder-data/data/processed/memes-test.ocr.json
  count: 559
  keys : ['category', 'img_captions', 'img_fname', 'meme_captions', 'metaphors', 'ocr_confidence', 'ocr_engine', 'ocr_text', 'post_id', 'title', 'url']


## 7. Verify Local Meme Images

Images should already exist under `data/raw/memes/memes/memes/`. This check verifies that JSON `img_fname` values resolve to local files before running the model.

If many files are missing, fix the Drive upload path. URL download is available only as a commented fallback in the next cell.


In [14]:
import json
from pathlib import Path

def check_images(json_path, limit=20):
    records = json.loads(Path(json_path).read_text(encoding="utf-8"))
    missing = []
    checked = 0
    for record in records[:limit]:
        fname = record.get("img_fname", "")
        path = IMAGE_CACHE_DIR / fname
        checked += 1
        if not path.exists():
            missing.append(str(path))
    print(f"Checked {checked} examples from {Path(json_path).name}")
    print(f"Missing: {len(missing)}")
    for path in missing[:10]:
        print("  missing:", path)

check_images(PROCESSED_DIR / "memes-trainval.ocr.json", limit=20)
check_images(PROCESSED_DIR / "memes-test.ocr.json", limit=20)


/content/meme-decoder
memes-trainval.ocr.json: 100% 5/5 [00:24<00:00,  4.83s/it]
memes-test.ocr.json: 100% 5/5 [00:00<00:00, 567.81it/s]
Finished image cache. Success: 10; failed: 0


In [ ]:
# Optional fallback only. Do not run this if images are already uploaded to Drive.
# It downloads missing images from each record's URL and caches them under RAW_DIR.
#
# %cd /content/meme-decoder
# !python -m scripts.download_images --processed-dir "$PROCESSED_DIR" --raw-dir "$RAW_DIR"


## 8. Zero-Shot Smoke Test

Input setting `4` means: image + title + image captions + OCR.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy zero-shot \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --eval-max-samples 20


## 9. Projector-Only Smoke Training

This automatically creates 5 fold files from `memes-trainval.ocr.json` if they do not exist.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 4 \
  --eval-max-samples 20


## 10. Projector + LoRA Smoke Training

For T4 GPUs, keep `batch-size=1`, use gradient accumulation, and start with `lora-r=8`.

In [ ]:
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --epochs 1 \
  --batch-size 1 \
  --grad-accum-steps 4 \
  --lora-r 8 \
  --eval-max-samples 20


## 11. Full Training Examples

Run these after the smoke tests pass.

In [ ]:
# Full projector-only training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-only \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 4


In [ ]:
# Full projector + LoRA training, setting 4
%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode train \
  --strategy projector-lora \
  --input-setting 4 \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --max-new-tokens 64 \
  --max-pixels 524288 \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum-steps 4 \
  --lora-r 8


## 12. Evaluate A Trained Checkpoint

Use the checkpoint list cell to find the run name, then set `MODEL_PATH`.

In [ ]:
from pathlib import Path

ckpt_root = OUTPUT_DIR / "checkpoints"
if ckpt_root.exists():
    for path in sorted(ckpt_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)[:10]:
        print(path)
else:
    print("No checkpoints yet:", ckpt_root)

In [ ]:
# Change this to one of the paths printed above.
MODEL_PATH = str(OUTPUT_DIR / "checkpoints" / "<run_name>")
STRATEGY = "projector-lora"  # or "projector-only"
INPUT_SETTING = 4

%cd /content/meme-decoder
!python -m scripts.run_pipeline \
  --mode eval \
  --strategy "$STRATEGY" \
  --input-setting "$INPUT_SETTING" \
  --model-path "$MODEL_PATH" \
  --checkpoint-name best \
  --processed-dir "$PROCESSED_DIR" \
  --raw-dir "$RAW_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --eval-json memes-test.ocr.json \
  --max-new-tokens 64 \
  --max-pixels 524288


## 13. Optional: Zero-Shot Input Ablation 1-5

In [ ]:
%cd /content/meme-decoder
for setting in [1, 2, 3, 4, 5]:
    print("\n=== zero-shot setting", setting, "===")
    !python -m scripts.run_pipeline \
      --mode eval \
      --strategy zero-shot \
      --input-setting "$setting" \
      --processed-dir "$PROCESSED_DIR" \
      --raw-dir "$RAW_DIR" \
      --output-dir "$OUTPUT_DIR" \
      --eval-json memes-test.ocr.json \
      --max-new-tokens 64 \
      --max-pixels 524288 \
      --eval-max-samples 50
